# Kaggle – Data on the top
Tu profe ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Métrica: RMSE

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

Donde $y_i$ es el valor real y $\hat{y}_i$ es el valor predicho. **Cuanto menor, mejor.**

---
# PARTE 1: Entrenamiento del modelo

## 1. Librerías

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Datos

In [ ]:
df = pd.read_csv('./data/train.csv', encoding='latin-1')

### 2.1 Exploración de los datos

In [ ]:
df.head()
df.info()
df.describe()
df.isnull().sum()
df.nunique()


### 2.2 Definir X e y


In [ ]:
def feature_engineer(data):
    """Convierte las columnas 'sucias' del dataset en features numéricas/categóricas limpias."""
    data = data.copy()

    # Ram: '8GB' -> 8
    data['Ram'] = data['Ram'].astype(str).str.replace('GB', '', regex=False).astype(int)

    # Weight: '1.86kg' -> 1.86
    data['Weight'] = data['Weight'].astype(str).str.replace('kg', '', regex=False).astype(float)

    # ScreenResolution: touchscreen, IPS, resolución y densidad de píxeles (PPI)
    data['Touchscreen'] = data['ScreenResolution'].str.contains('Touchscreen').astype(int)
    data['IPS'] = data['ScreenResolution'].str.contains('IPS').astype(int)
    res = data['ScreenResolution'].str.extract(r'(\d+)x(\d+)')
    data['ScreenWidth'] = res[0].astype(int)
    data['ScreenHeight'] = res[1].astype(int)
    data['PPI'] = ((data['ScreenWidth']**2 + data['ScreenHeight']**2)**0.5) / data['Inches']

    # Cpu: marca/gama y velocidad en GHz
    data['CpuBrand'] = data['Cpu'].apply(lambda x: ' '.join(x.split()[:2]) if 'Intel' in x else x.split()[0])
    data['CpuSpeedGHz'] = data['Cpu'].str.extract(r'([\d\.]+)GHz').astype(float)

    # Memory: puede tener varias unidades combinadas ("256GB SSD +  1TB HDD")
    def parse_memory(mem):
        mem = mem.replace('GB', '').replace('TB', '000')
        ssd = hdd = flash = hybrid = 0.0
        for part in mem.split('+'):
            part = part.strip()
            num = float(''.join(c for c in part if (c.isdigit() or c == '.')))
            if 'SSD' in part:
                ssd += num
            elif 'HDD' in part:
                hdd += num
            elif 'Flash' in part:
                flash += num
            elif 'Hybrid' in part:
                hybrid += num
        return pd.Series([ssd, hdd, flash, hybrid])

    data[['SSD_GB', 'HDD_GB', 'Flash_GB', 'Hybrid_GB']] = data['Memory'].apply(parse_memory)

    # Gpu: solo la marca (Intel / Nvidia / AMD / ARM)
    data['GpuBrand'] = data['Gpu'].apply(lambda x: x.split()[0])

    # Columnas ya no necesarias: son texto libre o quedaron reemplazadas por las nuevas features
    data = data.drop(columns=['ScreenResolution', 'Cpu', 'Memory', 'Gpu', 'Product'])

    return data


df_fe = feature_engineer(df)

X = df_fe.drop(columns=['laptop_ID', 'Price_in_euros'])
y = df_fe['Price_in_euros']

cat_cols = X.select_dtypes(include=['object', 'str']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object', 'str']).columns.tolist()

print('Columnas categóricas:', cat_cols)
print('Columnas numéricas:', num_cols)
X.head()

### 2.3 Dividir en train y test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Train:', X_train.shape, ' Test:', X_test.shape)

## 3. Procesado de datos

> 🚨 **Data leakage:** si usas un scaler, haz **`.fit()` SOLO sobre `X_train`** y luego aplica `.transform()` sobre `X_train` y `X_test` por separado.
>
> Recuerda también que **todo lo que hagas aquí deberás replicarlo después en `test.csv`** (sección 6).

In [ ]:
# One-Hot Encoding para las categóricas (fit SOLO sobre X_train)
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(X_train[cat_cols])

# Escalado para las numéricas (fit SOLO sobre X_train) — útil sobre todo para el SVR
scaler = StandardScaler()
scaler.fit(X_train[num_cols])


def preprocess(X_part):
    """Aplica SIEMPRE .transform() (nunca .fit) para evitar data leakage."""
    cat_arr = ohe.transform(X_part[cat_cols])
    cat_df = pd.DataFrame(
        cat_arr, columns=ohe.get_feature_names_out(cat_cols), index=X_part.index
    )

    num_arr = scaler.transform(X_part[num_cols])
    num_df = pd.DataFrame(num_arr, columns=num_cols, index=X_part.index)

    return pd.concat([num_df, cat_df], axis=1)


X_train_proc = preprocess(X_train)
X_test_proc = preprocess(X_test)

X_train_proc.shape, X_test_proc.shape

## 4. Modelado

### 4.1 Entrenamiento

In [ ]:
rf_model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf_model.fit(X_train_proc, y_train)

svr_model = SVR(kernel='rbf', C=1000, epsilon=10)
svr_model.fit(X_train_proc, y_train)

### 4.2 Métricas

Recuerda que en la competición se evalúa con **RMSE**.

In [ ]:
for name, model in [('RandomForest', rf_model), ('SVR', svr_model)]:
    preds = model.predict(X_test_proc)
    rmse = root_mean_squared_error(y_test, preds)
    print(f'{name:12s} -> RMSE: {rmse:.2f} €')

### 4.3 Optimización (up to you 🫰🏻)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [200, 400, 600],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1,
)
grid_search.fit(X_train_proc, y_train)

print('Mejores parámetros:', grid_search.best_params_)
print('Mejor RMSE (CV):', -grid_search.best_score_)

best_model = grid_search.best_estimator_
preds = best_model.predict(X_test_proc)
print('RMSE en test local:', root_mean_squared_error(y_test, preds))

## 5. Reentrenamiento sobre todos los datos de `train.csv`

Una vez afinado el modelo, reentrenamos con **todos** los datos disponibles antes de predecir sobre `test.csv`.

> ¿Por qué? El split anterior era solo para validar localmente. Para la submission final queremos aprovechar el 100% de los datos de entrenamiento.

In [ ]:
# Reajustamos encoder y scaler con TODO train.csv...
ohe_final = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe_final.fit(X[cat_cols])

scaler_final = StandardScaler()
scaler_final.fit(X[num_cols])


def preprocess_final(X_part):
    cat_arr = ohe_final.transform(X_part[cat_cols])
    cat_df = pd.DataFrame(
        cat_arr, columns=ohe_final.get_feature_names_out(cat_cols), index=X_part.index
    )
    num_arr = scaler_final.transform(X_part[num_cols])
    num_df = pd.DataFrame(num_arr, columns=num_cols, index=X_part.index)
    return pd.concat([num_df, cat_df], axis=1)


X_full_proc = preprocess_final(X)

# ...y reentrenamos el modelo ganador (RandomForest optimizado) con el 100% de los datos
final_model = RandomForestRegressor(**grid_search.best_params_, random_state=42, n_jobs=-1)
final_model.fit(X_full_proc, y)

---
# PARTE 2: Predicción y submission

Una vez tengas el modelo listo, toca predecir sobre `test.csv` y generar el archivo de submission.

## 6. Carga los datos de `test.csv`

In [ ]:
X_pred = pd.read_csv('./data/test.csv', encoding='latin-1')
X_pred.head()

## 7. Replica el procesado en `test.csv`

> ⚠️ Usa `.transform()`, **nunca `.fit_transform()`** sobre los datos de test.
>
> Lo único que **no puedes hacer** es eliminar filas.

In [ ]:
X_pred_fe = feature_engineer(X_pred)

# Nos aseguramos de no perder ninguna fila (392 en test.csv)
assert len(X_pred_fe) == len(X_pred)

X_pred_features = X_pred_fe.drop(columns=['laptop_ID'])

# ¡SOLO transform! El encoder y el scaler ya están "fit" con train.csv
X_pred_proc = preprocess_final(X_pred_features)

# Nos aseguramos de que las columnas coincidan exactamente con las de entrenamiento
X_pred_proc = X_pred_proc.reindex(columns=X_full_proc.columns, fill_value=0)

X_pred_proc.shape

## 8. Genera la submission

### 8.1 ¿Qué formato espera Kaggle?

In [ ]:
sample = pd.read_csv('./data/sample_submission.csv', encoding='latin-1')
sample.head()

### 8.2 Crea tu submission

In [ ]:
predicted_prices = final_model.predict(X_pred_proc)

submission = pd.DataFrame({
    'laptop_ID': X_pred_fe['laptop_ID'],
    'Price_in_euros': predicted_prices
})

submission.head()

### 8.3 Chequeador

Pásale el chequeador antes de subir a Kaggle. Si todo está bien, guardará el CSV automáticamente con un nombre único.

In [ ]:
def checker(df_to_submit, sample, filename=None):
    """
    Valida que tu submission tenga la forma requerida por Kaggle.
    Si es correcta, guarda el CSV listo para subir.
    Si no, lee el mensaje de error y corrígelo.
    """
    if df_to_submit.shape != sample.shape:
        print(' Shape incorrecto.')
        print(f'   Tu submission: {df_to_submit.shape} | Esperado: {sample.shape}')
        print('   Revisa que no hayas borrado filas del test ni añadido/quitado columnas.')
        return

    if not (df_to_submit.columns == sample.columns).all():
        print(' Nombres de columnas incorrectos.')
        print(f'   Tus columnas:       {list(df_to_submit.columns)}')
        print(f'   Columnas esperadas: {list(sample.columns)}')
        return

    if not (df_to_submit['laptop_ID'] == sample['laptop_ID']).all():
        print(' Los IDs no coinciden con sample_submission. Revisa que no hayas reordenado el test.csv.')
        return

    if filename is None:
        from datetime import datetime
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f'submission_{timestamp}.csv'

    df_to_submit.to_csv(filename, index=False)
    print(f" ¡Todo correcto! Submission guardada como '{filename}'. ¡A Kaggle!")

In [ ]:
checker(submission, sample)